# N1 — Camera–LiDAR Projection & Back-Projection

This notebook demonstrates how to project between 3D LiDAR point clouds and 2D camera images using the [KITTI Vision Benchmark](http://www.cvlibs.net/datasets/kitti/) dataset. Sensor fusion begins with geometry — before we can combine what a camera sees with what a LiDAR measures, we need to map points between their coordinate frames.

**What you will learn:**
1. The pinhole camera model and the role of intrinsic vs extrinsic calibration
2. How to project 3D world points into 2D image pixels (forward projection)
3. How to recover 3D world coordinates from image pixels under a ground-plane constraint (back-projection)
4. How small calibration errors propagate into large positional errors at distance

**Pipeline overview:**

```
         FORWARD  (3D → 2D)                    BACKWARD  (2D → 3D)
                                                 + ground-plane constraint
3D LiDAR Point (X,Y,Z,1)                   2D Pixel (u, v)
        │                                        │
        ▼                                        ▼
  Tr_velo_to_cam                            M⁻¹ with Z = z_ground
        │                                        │
        ▼                                        ▼
     R0_rect                                3D World Point (X, Y, z_ground)
        │
        ▼
       P2 = K · [I | t]
        │
        ▼
  2D Pixel (u, v) + depth
```

## 1. Install and Import Dependencies

In [ ]:
%pip install pykitti numpy matplotlib opencv-python-headless --quiet
%pip install "transformers==4.46.3" --quiet

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import cv2
import pykitti
import os

print("All libraries loaded successfully!")

## The Pinhole Camera Model

The pinhole camera model describes how 3D world points project to 2D image pixels through two transformations:

**Extrinsic parameters [R | t]** — a rigid-body transform (rotation + translation) from world coordinates to the camera's coordinate frame:

$$\mathbf{X}_\text{cam} = R \cdot \mathbf{X}_\text{world} + \mathbf{t}$$

**Intrinsic matrix K** — maps 3D points in camera coordinates to pixel coordinates:

$$K = \begin{bmatrix} f_x & 0 & c_x \\ 0 & f_y & c_y \\ 0 & 0 & 1 \end{bmatrix}$$

where $(f_x, f_y)$ are focal lengths in pixels and $(c_x, c_y)$ is the principal point (roughly the image center).

**Full projection** (homogeneous coordinates):

$$s \begin{bmatrix} u \\ v \\ 1 \end{bmatrix} = K \cdot [R \mid \mathbf{t}] \cdot \begin{bmatrix} X \\ Y \\ Z \\ 1 \end{bmatrix}$$

Divide by $s$ to get pixel coordinates $(u, v)$. **Depth information is lost** — infinitely many 3D points along a ray map to the same pixel. Recovering 3D requires an additional constraint (e.g., known ground plane) or a second sensor (e.g., LiDAR).

### Coordinate Frames in KITTI

| Frame | X | Y | Z | Origin |
|-------|---|---|---|--------|
| **Velodyne (LiDAR)** | Forward | Left | Up | LiDAR sensor |
| **Camera** | Right | Down | Forward | Camera optical center |
| **Image** | Right (u) | Down (v) | — | Top-left pixel |

The extrinsic transform `Tr_velo_to_cam` rotates and translates from the Velodyne frame to the camera frame — this handles the axis swap (LiDAR-forward → camera-depth) and the physical offset between the two sensors mounted on the vehicle.

## 2. Download KITTI Sample Data

The KITTI dataset provides synchronized camera images, LiDAR point clouds, and calibration files. We'll download a small sample drive for this demo.

The dataset is organized as:
```
kitti_data/
├── 2011_09_26/
│   ├── 2011_09_26_drive_0005_sync/
│   │   ├── image_02/data/          (left color camera)
│   │   ├── image_03/data/          (right color camera)
│   │   └── velodyne_points/data/   (LiDAR scans)
│   ├── calib_cam_to_cam.txt
│   ├── calib_imu_to_velo.txt
│   └── calib_velo_to_cam.txt
```

In [ ]:
import urllib.request
import zipfile
import shutil

KITTI_BASE = "kitti_data"
DATE = "2011_09_26"
DRIVE = "0005"

calib_url = f"https://s3.eu-central-1.amazonaws.com/avg-kitti/raw_data/{DATE}_calib.zip"
drive_url = f"https://s3.eu-central-1.amazonaws.com/avg-kitti/raw_data/{DATE}_drive_{DRIVE}/{DATE}_drive_{DRIVE}_sync.zip"

def download_and_extract(url, extract_to, description):
    zip_name = url.split("/")[-1]
    zip_path = os.path.join(extract_to, zip_name)

    target_check = os.path.join(extract_to, DATE)
    if description == "drive data":
        target_check = os.path.join(extract_to, DATE, f"{DATE}_drive_{DRIVE}_sync")

    if os.path.exists(target_check):
        print(f"  {description} already exists, skipping download.")
        return

    os.makedirs(extract_to, exist_ok=True)
    print(f"  Downloading {description} ...")
    urllib.request.urlretrieve(url, zip_path)
    print(f"  Extracting ...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_to)
    os.remove(zip_path)
    print(f"  Done.")

print("Downloading KITTI sample data...\n")
download_and_extract(calib_url, KITTI_BASE, "calibration files")
download_and_extract(drive_url, KITTI_BASE, "drive data")

print(f"\nData ready at: {os.path.abspath(KITTI_BASE)}")

## 3. Load Data with pykitti

The `pykitti` library handles parsing the KITTI directory structure and calibration files. It gives us direct access to:
- **Camera images** (cam2 = left color camera)
- **Velodyne point clouds** (N x 4 arrays: X, Y, Z, reflectance)
- **Calibration matrices** (camera intrinsics, velodyne-to-camera extrinsics)

In [ ]:
data = pykitti.raw(KITTI_BASE, DATE, DRIVE)

FRAME_IDX = 0

img = np.array(data.get_cam2(FRAME_IDX))
velo = data.get_velo(FRAME_IDX)  # (N, 4) -> X, Y, Z, reflectance

print(f"Image shape:        {img.shape}  (H x W x C)")
print(f"Point cloud shape:  {velo.shape}  (N points x [X, Y, Z, reflectance])")
print(f"Point cloud range:  X [{velo[:,0].min():.1f}, {velo[:,0].max():.1f}]  "
      f"Y [{velo[:,1].min():.1f}, {velo[:,1].max():.1f}]  "
      f"Z [{velo[:,2].min():.1f}, {velo[:,2].max():.1f}]")

In [ ]:
# Display the raw camera image
fig, ax = plt.subplots(1, 1, figsize=(14, 5))
ax.imshow(img)
ax.set_title("Camera 2 (Left Color) — Raw Image", fontsize=14)
ax.axis("off")
plt.tight_layout()
plt.show()

## 4. Visualize the Raw 3D Point Cloud

Before projecting onto the image, let's look at the raw LiDAR scan from a bird's-eye view (top-down). Each point has an X (forward), Y (left), Z (up), and reflectance value.

In [ ]:
# Bird's-eye view of the point cloud
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Top-down view (X forward, Y left)
ax = axes[0]
scatter = ax.scatter(velo[:, 0], velo[:, 1], c=velo[:, 2], cmap="viridis",
                     s=0.3, alpha=0.5, vmin=-2, vmax=1)
ax.set_xlabel("X (forward, m)")
ax.set_ylabel("Y (left, m)")
ax.set_title("Bird's-Eye View (colored by height Z)")
ax.set_aspect("equal")
ax.set_xlim(-10, 80)
ax.set_ylim(-40, 40)
plt.colorbar(scatter, ax=ax, label="Z (m)")

# Front view (Y left, Z up)
ax = axes[1]
scatter = ax.scatter(velo[:, 1], velo[:, 2], c=velo[:, 0], cmap="plasma",
                     s=0.3, alpha=0.5, vmin=0, vmax=50)
ax.set_xlabel("Y (left, m)")
ax.set_ylabel("Z (up, m)")
ax.set_title("Front View (colored by depth X)")
ax.set_aspect("equal")
ax.set_xlim(-30, 30)
ax.set_ylim(-3, 10)
plt.colorbar(scatter, ax=ax, label="X (m)")

plt.tight_layout()
plt.show()

print(f"Total points in scan: {len(velo):,}")

## 5. Understanding the Calibration Matrices

To project a 3D LiDAR point onto the 2D camera image, we chain three transformations:

| Matrix | Shape | Purpose |
|--------|-------|---------|
| **Tr_velo_to_cam** | 4×4 | Rigid-body transform from Velodyne (LiDAR) coordinates to the reference camera frame |
| **R0_rect** | 4×4 | Stereo rectification rotation (aligns camera axes), provided as 4×4 by pykitti |
| **P2** | 3×4 | Camera 2 projection matrix (intrinsics + baseline offset) |

The full projection for a homogeneous 3D point **X** = (X, Y, Z, 1):

$$\mathbf{y} = P_2 \cdot R_{0,\text{rect}} \cdot T_{\text{velo→cam}} \cdot \mathbf{X}$$

The result **y** = (u·z, v·z, z) — divide by z to get pixel coordinates (u, v) and depth.

In [ ]:
# Extract calibration matrices from pykitti
# T_cam0_velo: 4x4 rigid body transform (velo -> cam0)
T_cam_velo = np.array(data.calib.T_cam0_velo)

# R0_rect: rectifying rotation (pykitti returns this as 4x4)
R0_rect = np.array(data.calib.R_rect_00)

# Ensure R0_rect is 4x4 for use in matrix chains
if R0_rect.shape == (3, 3):
    R0_ext = np.eye(4)
    R0_ext[:3, :3] = R0_rect
else:
    R0_ext = R0_rect

# P2: 3x4 projection matrix for camera 2 (left color)
P2 = np.array(data.calib.P_rect_20)

print(f"Tr_velo_to_cam  {T_cam_velo.shape}:")
print(T_cam_velo)
print(f"\nR0_rect  {R0_rect.shape} → R0_ext  {R0_ext.shape}:")
print(R0_ext)
print(f"\nP2 — Camera 2 projection  {P2.shape}:")
print(P2)

In [ ]:
# Extract the intrinsic matrix K from P2
# In KITTI, P2 = K @ [I | t] where t accounts for the stereo baseline offset to camera 2
K = P2[:3, :3]
t_cam2 = np.linalg.solve(K, P2[:, 3])

print("Intrinsic matrix K (extracted from P2):")
print(K)
print(f"\nFocal length:    fx = {K[0,0]:.1f} px,  fy = {K[1,1]:.1f} px")
print(f"Principal point: cx = {K[0,2]:.1f} px,  cy = {K[1,2]:.1f} px")
print(f"Baseline offset: t  = [{t_cam2[0]:.4f}, {t_cam2[1]:.4f}, {t_cam2[2]:.4f}] m")

### What's Inside the Calibration Files?

The KITTI calibration directory stores three text files — each describes the spatial relationship between a pair of sensors as a **rotation R** (3×3) and a **translation T** (3×1):

| File | Transform | Purpose |
|------|-----------|---------|
| `calib_velo_to_cam.txt` | Velodyne LiDAR → Camera 0 | The extrinsic transform used for LiDAR-to-image projection |
| `calib_imu_to_velo.txt` | IMU/GPS → Velodyne LiDAR | Connects inertial navigation to the LiDAR frame |
| `calib_cam_to_cam.txt` | Camera 0 → Cameras 0–3 | Per-camera intrinsics, distortion, and stereo rectification |

Together, R and T form a 4×4 **homogeneous transform matrix** that converts a 3D point from one sensor's coordinate frame to another:

$$T = \begin{bmatrix} R & \mathbf{t} \\ \mathbf{0}^T & 1 \end{bmatrix}$$

Let's read the raw files and see what the numbers mean physically.

In [ ]:
def parse_kitti_rigid_calib(filepath):
    """Parse a KITTI calibration file containing R (3×3) and T (3×1)."""
    fields = {}
    with open(filepath) as f:
        for line in f:
            if ':' in line:
                key, val = line.strip().split(':', 1)
                fields[key.strip()] = val.strip()
    R = np.array([float(x) for x in fields['R'].split()]).reshape(3, 3)
    T = np.array([float(x) for x in fields['T'].split()])
    return R, T


# ── Parse and display calib_velo_to_cam.txt ──
calib_vc_path = os.path.join(KITTI_BASE, DATE, "calib_velo_to_cam.txt")

print("File: calib_velo_to_cam.txt")
print("═" * 70)
with open(calib_vc_path) as f:
    for line in f:
        print(f"  {line.rstrip()}")
print("═" * 70)

R_vc_file, T_vc_file = parse_kitti_rigid_calib(calib_vc_path)

print(f"\nR — Rotation (3×3): rotates vectors from Velodyne into Camera axes\n")
for row in R_vc_file:
    print(f"  [{row[0]:13.7f}  {row[1]:13.7f}  {row[2]:13.7f}]")

print(f"\nT — Translation (3×1): Velodyne origin in Camera coordinates\n")
print(f"  [{T_vc_file[0]:.7f},  {T_vc_file[1]:.7f},  {T_vc_file[2]:.7f}] m")
print(f"  → Velodyne is {abs(T_vc_file[0])*100:.1f} cm {'right of' if T_vc_file[0]>0 else 'left of'}, "
      f"{abs(T_vc_file[1])*100:.1f} cm {'below' if T_vc_file[1]>0 else 'above'}, "
      f"and {abs(T_vc_file[2])*100:.1f} cm {'in front of' if T_vc_file[2]>0 else 'behind'} the camera")

Tr_vc_file = np.eye(4)
Tr_vc_file[:3, :3] = R_vc_file
Tr_vc_file[:3, 3] = T_vc_file

print(f"\n4×4 Homogeneous Transform  [R | T ; 0 0 0 1]:\n")
for i in range(4):
    print(f"  [{'  '.join(f'{Tr_vc_file[i,j]:13.7f}' for j in range(4))}]")

print(f"\nNote: pykitti premultiplies this by R_rect_00 (stereo rectification)")
print(f"to produce the T_cam0_velo used in the projection pipeline above.")

### IMU → Velodyne Calibration

The IMU (Inertial Measurement Unit) provides GPS position, velocity, and orientation. Both the IMU and Velodyne use the same axis convention (X-forward, Y-left, Z-up), so R is nearly identity — the transform is dominated by the **translation** between the two physically separated sensors. The IMU sits inside the vehicle while the Velodyne is mounted on the roof rack above.

In [ ]:
# ── Parse and display calib_imu_to_velo.txt ──
calib_iv_path = os.path.join(KITTI_BASE, DATE, "calib_imu_to_velo.txt")

print("File: calib_imu_to_velo.txt")
print("═" * 70)
with open(calib_iv_path) as f:
    for line in f:
        print(f"  {line.rstrip()}")
print("═" * 70)

R_iv_file, T_iv_file = parse_kitti_rigid_calib(calib_iv_path)

print(f"\nR — Rotation (3×3): rotates vectors from IMU into Velodyne axes\n")
for row in R_iv_file:
    print(f"  [{row[0]:13.7f}  {row[1]:13.7f}  {row[2]:13.7f}]")

angle_iv = np.degrees(np.arccos(np.clip((np.trace(R_iv_file) - 1) / 2, -1, 1)))
print(f"\n  Rotation angle: {angle_iv:.2f}° (nearly identity — IMU and Velodyne axes are almost aligned)")

print(f"\nT — Translation (3×1): IMU origin in Velodyne coordinates\n")
print(f"  [{T_iv_file[0]:.7f},  {T_iv_file[1]:.7f},  {T_iv_file[2]:.7f}] m")
print(f"  → IMU is {abs(T_iv_file[0])*100:.1f} cm {'forward of' if T_iv_file[0]>0 else 'behind'}, "
      f"{abs(T_iv_file[1])*100:.1f} cm {'left of' if T_iv_file[1]>0 else 'right of'}, "
      f"and {abs(T_iv_file[2])*100:.1f} cm {'above' if T_iv_file[2]>0 else 'below'} the Velodyne")

Tr_iv_file = np.eye(4)
Tr_iv_file[:3, :3] = R_iv_file
Tr_iv_file[:3, 3] = T_iv_file

print(f"\n4×4 Homogeneous Transform  [R | T ; 0 0 0 1]:\n")
for i in range(4):
    print(f"  [{'  '.join(f'{Tr_iv_file[i,j]:13.7f}' for j in range(4))}]")

# ── Combined visualization: all three sensor frames in Velodyne coordinates ──
cam_origin = -R_vc_file.T @ T_vc_file   # Camera 0 position in Velodyne frame
cam_axes = R_vc_file.T                   # Camera 0 axes in Velodyne frame
imu_origin = T_iv_file.copy()            # IMU position in Velodyne frame
imu_axes = R_iv_file.copy()              # IMU axes in Velodyne frame

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

arrow_len = 0.12
rgb = ['#d62728', '#2ca02c', '#1f77b4']

sensor_frames = [
    (np.zeros(3), np.eye(3), "VELODYNE",
     ["X (fwd)", "Y (left)", "Z (up)"], 'black'),
    (cam_origin, cam_axes, "CAMERA 0",
     ["X (right)", "Y (down)", "Z (fwd)"], '#1f77b4'),
    (imu_origin, imu_axes, "IMU / GPS",
     ["X (fwd)", "Y (left)", "Z (up)"], '#e67e22'),
]

for origin, axes_mat, name, labels, name_clr in sensor_frames:
    for i, (c, lbl) in enumerate(zip(rgb, labels)):
        d = axes_mat[:, i] * arrow_len
        ax.quiver(*origin, *d, color=c, linewidth=2, arrow_length_ratio=0.1)
        ax.text(*(origin + d * 1.35), lbl, color=c, fontsize=7, fontweight='bold')
    ax.scatter(*origin, color=name_clr, s=80, zorder=5, edgecolors='white', linewidths=0.5)
    ax.text(origin[0], origin[1], origin[2] - arrow_len * 0.5,
            name, fontsize=10, fontweight='bold', ha='center', color=name_clr)

# Dashed lines from Velodyne to each other sensor
for other in [cam_origin, imu_origin]:
    ax.plot(*zip(np.zeros(3), other), 'k--', linewidth=0.8, alpha=0.3)
    mid = other / 2
    dist = np.linalg.norm(other)
    ax.text(mid[0] + 0.03, mid[1] + 0.03, mid[2], f"{dist:.2f} m",
            fontsize=8, color='gray')

ax.set_xlabel('X — forward (m)')
ax.set_ylabel('Y — left (m)')
ax.set_zlabel('Z — up (m)')
ax.set_title('KITTI Sensor Coordinate Frames\n(all shown in Velodyne coordinates)', fontsize=13)
ax.view_init(elev=20, azim=-50)

all_origins = np.vstack([np.zeros(3), cam_origin, imu_origin])
center = all_origins.mean(axis=0)
span = max(np.ptp(all_origins, axis=0).max() * 0.65, 0.3)
for setter, c_val in zip([ax.set_xlim, ax.set_ylim, ax.set_zlim], center):
    setter(c_val - span, c_val + span)

plt.tight_layout()
plt.show()

print(f"\nDistances from Velodyne:")
print(f"  → Camera 0:  {np.linalg.norm(cam_origin):.3f} m")
print(f"  → IMU / GPS: {np.linalg.norm(imu_origin):.3f} m")

## 6. Project Point Cloud onto the Camera Image

Now we'll apply the full transformation pipeline to every LiDAR point and keep only the points that land inside the image frame.

In [ ]:
def project_velo_to_cam2(velo_points, T_cam_velo, R0_rect, P2, img_shape):
    """
    Project Velodyne points onto Camera 2 image plane.

    Returns:
        pts_2d: (M, 2) pixel coordinates [u, v]
        depths: (M,)   depth in camera frame
    """
    pts_3d = velo_points[:, :3]

    # Step 1: Filter points in front of the LiDAR (positive X in velodyne frame)
    front_mask = pts_3d[:, 0] > 0
    pts_3d = pts_3d[front_mask]

    # Step 2: Convert to homogeneous coordinates (N, 4)
    ones = np.ones((pts_3d.shape[0], 1))
    pts_3d_hom = np.hstack([pts_3d, ones])  # (N, 4)

    # Step 3: Velodyne -> Camera 0 reference frame  (4x4 @ 4xN = 4xN)
    pts_cam = T_cam_velo @ pts_3d_hom.T

    # Step 4: Apply rectification
    # R0_rect may be 3x3 or 4x4 depending on pykitti version — normalize to 4x4
    if R0_rect.shape == (3, 3):
        R0_ext = np.eye(4)
        R0_ext[:3, :3] = R0_rect
    else:
        R0_ext = R0_rect
    pts_cam_rect = R0_ext @ pts_cam  # (4, N)

    # Step 5: Project to image plane with P2  (3x4 @ 4xN = 3xN)
    pts_2d_hom = P2 @ pts_cam_rect

    # Step 6: Normalize by depth (divide by z)
    depths = pts_2d_hom[2, :]
    pts_2d = pts_2d_hom[:2, :] / depths  # (2, N)
    pts_2d = pts_2d.T  # (N, 2) -> [u, v]

    # Step 7: Filter to points within image boundaries and positive depth
    h, w = img_shape[:2]
    valid = (
        (depths > 0) &
        (pts_2d[:, 0] >= 0) & (pts_2d[:, 0] < w) &
        (pts_2d[:, 1] >= 0) & (pts_2d[:, 1] < h)
    )

    return pts_2d[valid], depths[valid]

pts_2d, depths = project_velo_to_cam2(velo, T_cam_velo, R0_rect, P2, img.shape)

print(f"Original points:   {len(velo):,}")
print(f"Projected & valid: {len(pts_2d):,}")
print(f"Depth range:       {depths.min():.1f} m  to  {depths.max():.1f} m")

## 7. Visualize: LiDAR Points Projected onto Camera Image

Each projected point is colored by its depth — closer objects appear warmer (red/yellow), farther objects appear cooler (blue/green).

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(16, 6))
ax.imshow(img)
scatter = ax.scatter(pts_2d[:, 0], pts_2d[:, 1],
                     c=depths, cmap="jet_r", s=1, alpha=0.7,
                     vmin=0, vmax=50)
plt.colorbar(scatter, ax=ax, label="Depth (m)", shrink=0.7)
ax.set_title("LiDAR Point Cloud Projected onto Camera 2 Image (colored by depth)", fontsize=14)
ax.axis("off")
plt.tight_layout()
plt.show()

## 8. Depth-Filtered Views

Let's create multiple views filtered by distance range. This highlights how LiDAR provides depth information that a camera alone cannot.

In [ ]:
depth_ranges = [
    (0, 10,  "Near (0–10 m)"),
    (10, 25, "Mid (10–25 m)"),
    (25, 80, "Far (25–80 m)"),
]

fig, axes = plt.subplots(len(depth_ranges), 1, figsize=(16, 5 * len(depth_ranges)))

for ax, (d_min, d_max, label) in zip(axes, depth_ranges):
    mask = (depths >= d_min) & (depths < d_max)
    ax.imshow(img)
    if mask.sum() > 0:
        ax.scatter(pts_2d[mask, 0], pts_2d[mask, 1],
                   c=depths[mask], cmap="jet_r", s=2, alpha=0.8,
                   vmin=d_min, vmax=d_max)
    ax.set_title(f"{label} — {mask.sum():,} points", fontsize=13)
    ax.axis("off")

plt.tight_layout()
plt.show()

## 2D → 3D Back-Projection: Finding the Drivable Surface

Forward projection loses depth — a pixel $(u, v)$ could correspond to any point along a ray. But if we **constrain** a point to lie on the ground plane ($Z_\text{velo} = z_\text{ground}$), we can solve for its 3D position.

**The practical challenge:** which pixels are actually road? In this KITTI scene the road surface blends into the sidewalk — simple edge detection (Canny, Sobel) won't reliably separate them. We need a model that understands *what* things are, not just where gradients exist. We'll use **SegFormer**, a lightweight semantic segmentation model trained on the Cityscapes urban driving dataset, to classify every pixel as road, sidewalk, building, vehicle, etc. — then back-project the road boundary to the ground plane.

### Back-Projection Math

Given the combined 3×4 projection matrix $M = P_2 \cdot R_{0} \cdot T_{\text{velo→cam}}$:

$$\begin{bmatrix} u \cdot s \\ v \cdot s \\ s \end{bmatrix} = M \cdot \begin{bmatrix} X \\ Y \\ z_\text{ground} \\ 1 \end{bmatrix}$$

Since $z_\text{ground}$ is known, eliminating $s$ yields a **2×2 linear system** solvable via Cramer's rule:

$$\begin{bmatrix} m_{00} - u \cdot m_{20} & m_{01} - u \cdot m_{21} \\ m_{10} - v \cdot m_{20} & m_{11} - v \cdot m_{21} \end{bmatrix} \begin{bmatrix} X \\ Y \end{bmatrix} = \begin{bmatrix} u \cdot c - a \\ v \cdot c - b \end{bmatrix}$$

where $a = m_{02} z_g + m_{03}$, $b = m_{12} z_g + m_{13}$, $c = m_{22} z_g + m_{23}$.

This is **inverse perspective mapping** — it only works for points on the assumed plane. Off-plane objects (vehicles, pedestrians) will be mis-localized.

In [ ]:
%pip install transformers torch torchvision --quiet

import torch
import torchvision
from transformers import SegformerForSemanticSegmentation, SegformerImageProcessor
from PIL import Image
import torch.nn.functional as F

# Load SegFormer-B0 — lightweight (~15 MB), trained on Cityscapes
model_name = "nvidia/segformer-b0-finetuned-cityscapes-1024-1024"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"   # use the H100 MIG slice if present
processor = SegformerImageProcessor.from_pretrained(model_name)
seg_model = SegformerForSemanticSegmentation.from_pretrained(model_name).to(DEVICE)
seg_model.eval()
print(f"SegFormer running on: {DEVICE}")

# Run inference (inputs must sit on the same device as the model)
inputs = processor(images=Image.fromarray(img), return_tensors="pt").to(DEVICE)
# autocast = FP16 (half precision) on GPU: ~2x faster here, negligible effect on an argmax mask
with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=(DEVICE == "cuda")):
    logits = seg_model(**inputs).logits

# Upsample to original image size and get per-pixel class
upsampled = F.interpolate(logits, size=img.shape[:2], mode="bilinear", align_corners=False)
seg_mask = upsampled.argmax(dim=1).squeeze().cpu().numpy()

CITYSCAPES_LABELS = [
    'road', 'sidewalk', 'building', 'wall', 'fence', 'pole',
    'traffic light', 'traffic sign', 'vegetation', 'terrain',
    'sky', 'person', 'rider', 'car', 'truck', 'bus', 'train',
    'motorcycle', 'bicycle'
]

road_mask = seg_mask == 0
sidewalk_mask = seg_mask == 1

# Visualization: original | segmentation overlay | road-only mask
overlay = img.copy()
overlay[road_mask] = [0, 200, 0]
overlay[sidewalk_mask] = [200, 0, 0]
blended = cv2.addWeighted(img, 0.5, overlay, 0.5, 0)

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
axes[0].imshow(img)
axes[0].set_title("Original Image", fontsize=13)
axes[0].axis("off")

axes[1].imshow(blended)
axes[1].set_title("Segmentation: Green = Road, Red = Sidewalk", fontsize=13)
axes[1].axis("off")

axes[2].imshow(road_mask, cmap="Greens")
axes[2].set_title("Road Surface Mask", fontsize=13)
axes[2].axis("off")

plt.tight_layout()
plt.show()

print(f"Road pixels:     {road_mask.sum():,}  ({road_mask.mean()*100:.1f}% of image)")
print(f"Sidewalk pixels: {sidewalk_mask.sum():,}  ({sidewalk_mask.mean()*100:.1f}% of image)")

### Back-Projecting the Road Surface to Bird's-Eye View

Now we take every road pixel from the segmentation, back-project it to the ground plane, and render a bird's-eye view oriented as if looking down from above the car — forward is up, left is left. This is the same perspective you'd see on a planning/mapping display.

In [ ]:
# Estimate ground-plane height from LiDAR points in front of the car
# In the Velodyne frame, Z is up and the sensor sits ~1.73 m above road level.
# We take forward-facing points near the car and use the 5th percentile of Z.
ground_candidates = velo[
    (velo[:, 0] > 3) & (velo[:, 0] < 25) &   # forward, not too far
    (np.abs(velo[:, 1]) < 6)                   # within a lane width
]
z_ground = np.percentile(ground_candidates[:, 2], 5)
print(f"Estimated ground plane: z_ground = {z_ground:.3f} m (Velodyne frame)")

# Combined projection matrix
M = P2 @ R0_ext @ T_cam_velo  # (3, 4)

def backproject_to_ground(pixels_uv, M, z_ground):
    """
    Vectorized back-projection of 2D pixels to 3D ground-plane coordinates.
    Uses Cramer's rule to solve the 2x2 system per pixel.
    """
    u = pixels_uv[:, 0].astype(np.float64)
    v = pixels_uv[:, 1].astype(np.float64)

    a = M[0, 2] * z_ground + M[0, 3]
    b = M[1, 2] * z_ground + M[1, 3]
    c = M[2, 2] * z_ground + M[2, 3]

    A00 = M[0, 0] - u * M[2, 0]
    A01 = M[0, 1] - u * M[2, 1]
    A10 = M[1, 0] - v * M[2, 0]
    A11 = M[1, 1] - v * M[2, 1]
    rhs0 = u * c - a
    rhs1 = v * c - b

    det = A00 * A11 - A01 * A10
    det[det == 0] = np.nan

    X = (rhs0 * A11 - rhs1 * A01) / det
    Y = (A00 * rhs1 - A10 * rhs0) / det
    return np.stack([X, Y, np.full_like(X, z_ground)], axis=1)

# --- Collect ALL road pixels (subsample for speed) ---
road_v, road_u = np.where(road_mask)
step = max(1, len(road_v) // 30000)
road_pixels = np.stack([road_u[::step], road_v[::step]], axis=1)
print(f"Road pixels to back-project: {len(road_pixels):,}  (subsampled from {len(road_v):,})")

# --- Back-project to ground plane ---
road_3d = backproject_to_ground(road_pixels, M, z_ground)

# Filter to physically reasonable range
reasonable = (
    np.isfinite(road_3d[:, 0]) &
    (road_3d[:, 0] > 0) & (road_3d[:, 0] < 80) &
    (np.abs(road_3d[:, 1]) < 25)
)
road_3d_valid = road_3d[reasonable]
road_px_valid = road_pixels[reasonable]

# Get the camera RGB color for each road pixel
road_colors = img[road_px_valid[:, 1].astype(int), road_px_valid[:, 0].astype(int)] / 255.0

# --- Visualization ---
fig, axes = plt.subplots(1, 2, figsize=(16, 9))

# Left: segmented road highlighted on image
ax = axes[0]
blended_road = img.copy()
blended_road[road_mask] = (0.5 * blended_road[road_mask] + 0.5 * np.array([0, 200, 0])).astype(np.uint8)
ax.imshow(blended_road)
ax.set_title("Segmented Road Surface", fontsize=13)
ax.axis("off")

# Right: bird's-eye view — forward (X) is UP, left-right (Y) is horizontal
ax = axes[1]
ax.scatter(road_3d_valid[:, 1], road_3d_valid[:, 0],
           c=road_colors, s=1, alpha=0.6)

# Draw ego vehicle marker at origin
ax.plot(0, 0, marker="^", color="red", markersize=14, zorder=10)
ax.annotate("EGO", (0, 0), textcoords="offset points", xytext=(10, -5),
            fontsize=9, fontweight="bold", color="red")

ax.set_xlabel("Y — lateral (m)", fontsize=12)
ax.set_ylabel("X — forward (m)", fontsize=12)
ax.set_title("Road Surface — Bird's-Eye View (back-projected)", fontsize=13)
ax.set_aspect("equal")
ax.set_xlim(15, -15)   # invert so left-in-world appears left-on-screen
ax.set_ylim(-5, 60)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Back-projected {len(road_3d_valid):,} road surface points to the ground plane.")

### Road Surface Detection — Bird's-Eye View Video

We can extend the single-frame back-projection to the full drive sequence: for each frame, SegFormer segments the road surface, and the road pixels are back-projected onto the ground plane. The result is a side-by-side video — camera view with road overlay on the left, metric bird's-eye view on the right — showing how the drivable surface evolves as the car moves through the scene.

In [ ]:
import io
import imageio.v3 as iio
import matplotlib

original_backend = matplotlib.get_backend()
matplotlib.use("Agg")

num_frames = min(len(data.cam2_files), 80)
bev_frames = []

print(f"Rendering {num_frames} frames: road segmentation → bird's-eye view...")
print(f"(ground plane z = {z_ground:.3f} m in Velodyne frame)\n")

for idx in range(num_frames):
    frame_img = np.array(data.get_cam2(idx))

    # Segment road surface with SegFormer
    inputs = processor(images=Image.fromarray(frame_img), return_tensors="pt").to(DEVICE)
    # autocast = FP16 (half precision) on GPU: ~2x faster here, negligible effect on an argmax mask
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=(DEVICE == "cuda")):
        logits = seg_model(**inputs).logits
    up = F.interpolate(logits, size=frame_img.shape[:2], mode="bilinear", align_corners=False)
    frame_road = up.argmax(dim=1).squeeze().cpu().numpy() == 0

    # Back-project road pixels to ground plane
    rv, ru = np.where(frame_road)
    if len(rv) > 0:
        step_bp = max(1, len(rv) // 20000)
        road_px = np.stack([ru[::step_bp], rv[::step_bp]], axis=1)
        road_3d_frame = backproject_to_ground(road_px, M, z_ground)

        keep = (
            np.isfinite(road_3d_frame[:, 0]) &
            (road_3d_frame[:, 0] > 0) & (road_3d_frame[:, 0] < 80) &
            (np.abs(road_3d_frame[:, 1]) < 25)
        )
        road_3d_v = road_3d_frame[keep]
        road_px_v = road_px[keep]
        road_clr = frame_img[road_px_v[:, 1].astype(int),
                             road_px_v[:, 0].astype(int)] / 255.0
    else:
        road_3d_v = np.empty((0, 3))
        road_clr = np.empty((0, 3))

    # ── Render side-by-side ──
    fig, axes = plt.subplots(1, 2, figsize=(16, 5), dpi=100,
                             gridspec_kw={"width_ratios": [2, 1]})

    # Left: camera image with road overlay
    blended = frame_img.copy()
    if frame_road.any():
        blended[frame_road] = (0.5 * blended[frame_road]
                               + 0.5 * np.array([0, 200, 0])).astype(np.uint8)
    axes[0].imshow(blended)
    axes[0].set_title(f"Road Segmentation — Frame {idx}", fontsize=12)
    axes[0].axis("off")

    # Right: bird's-eye view
    if len(road_3d_v) > 0:
        axes[1].scatter(road_3d_v[:, 1], road_3d_v[:, 0],
                        c=road_clr, s=1, alpha=0.6)
    axes[1].plot(0, 0, marker="^", color="red", markersize=12, zorder=10)
    axes[1].annotate("EGO", (0, 0), textcoords="offset points",
                     xytext=(8, -4), fontsize=8, fontweight="bold", color="red")
    axes[1].set_xlabel("Y — lateral (m)", fontsize=10)
    axes[1].set_ylabel("X — forward (m)", fontsize=10)
    axes[1].set_title("Bird's-Eye View (ground plane)", fontsize=12)
    axes[1].set_aspect("equal")
    axes[1].set_xlim(15, -15)
    axes[1].set_ylim(-5, 60)
    axes[1].grid(True, alpha=0.3)
    axes[1].set_facecolor("#f0f0f0")

    plt.tight_layout()

    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight")
    buf.seek(0)
    bev_frames.append(iio.imread(buf))
    plt.close(fig)

    if (idx + 1) % 20 == 0:
        print(f"  Rendered {idx + 1}/{num_frames} frames")

matplotlib.use(original_backend)

output_bev = os.path.join("videos", "road_bev_video.mp4")
os.makedirs("videos", exist_ok=True)
iio.imwrite(output_bev, bev_frames, fps=10, codec="libx264")
print(f"\nVideo saved: {output_bev}  ({num_frames} frames, 10 fps)")

## Dense Depth Map from Sparse LiDAR

The LiDAR gives us depth at ~20k scattered pixels — we can interpolate to fill the entire image. Linear interpolation provides smooth depth transitions where data is dense, and nearest-neighbor fills the remaining gaps.

In [ ]:
from scipy.interpolate import griddata

h, w = img.shape[:2]

# Sparse LiDAR points as (u, v) with depth values
points = np.column_stack([pts_2d[:, 0], pts_2d[:, 1]])
values = depths

# Interpolate: linear where data is dense, nearest-neighbor to fill the rest
grid_u, grid_v = np.meshgrid(np.arange(w), np.arange(h))
dense_linear = griddata(points, values, (grid_u, grid_v), method="linear")
dense_nearest = griddata(points, values, (grid_u, grid_v), method="nearest")
dense_depth = np.where(np.isnan(dense_linear), dense_nearest, dense_linear)

fig, axes = plt.subplots(1, 2, figsize=(18, 5))

axes[0].imshow(img)
axes[0].set_title("Original Camera Image", fontsize=13)
axes[0].axis("off")

depth_vis = axes[1].imshow(dense_depth, cmap="jet_r", vmin=0, vmax=50)
axes[1].set_title("Dense Depth Map (interpolated from LiDAR)", fontsize=13)
axes[1].axis("off")
plt.colorbar(depth_vis, ax=axes[1], label="Depth (m)", shrink=0.7)

plt.tight_layout()
plt.show()

## 10. Animate Across Multiple Frames

Let's project LiDAR data across several frames in the drive sequence to see the fusion in motion.

In [ ]:
from IPython import display as ipd
import time

sample_frames = range(0, min(len(data.cam2_files), 50), 1)

fig, ax = plt.subplots(1, 1, figsize=(16, 6))

for idx in sample_frames:
    frame_img = np.array(data.get_cam2(idx))
    frame_velo = data.get_velo(idx)

    pts, d = project_velo_to_cam2(frame_velo, T_cam_velo, R0_rect, P2, frame_img.shape)

    ax.clear()
    ax.imshow(frame_img)
    if len(pts) > 0:
        ax.scatter(pts[:, 0], pts[:, 1], c=d, cmap="jet_r", s=1, alpha=0.7, vmin=0, vmax=50)
    ax.set_title(f"Frame {idx} — {len(pts):,} projected points", fontsize=13)
    ax.axis("off")

    ipd.clear_output(wait=True)
    ipd.display(fig)
    time.sleep(0.1)

plt.close()
print("Animation complete.")

## 11. Export as MP4 Video

Save the LiDAR-camera fusion animation as a video file for sharing.

In [ ]:
%pip install imageio[ffmpeg] --quiet

import io
import imageio.v3 as iio
import matplotlib

original_backend = matplotlib.get_backend()
matplotlib.use("Agg")

num_frames = min(len(data.cam2_files), 80)
frames = []

for idx in range(num_frames):
    frame_img = np.array(data.get_cam2(idx))
    frame_velo = data.get_velo(idx)
    pts, d = project_velo_to_cam2(frame_velo, T_cam_velo, R0_rect, P2, frame_img.shape)

    fig, ax = plt.subplots(1, 1, figsize=(14, 5), dpi=100)
    ax.imshow(frame_img)
    if len(pts) > 0:
        ax.scatter(pts[:, 0], pts[:, 1], c=d, cmap="jet_r", s=1, alpha=0.7, vmin=0, vmax=50)
    ax.set_title(f"LiDAR-Camera Fusion — Frame {idx}", fontsize=12)
    ax.axis("off")
    plt.tight_layout()

    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight")
    buf.seek(0)
    frames.append(iio.imread(buf))
    plt.close(fig)

    if (idx + 1) % 20 == 0:
        print(f"  Rendered {idx + 1}/{num_frames} frames")

matplotlib.use(original_backend)

output_video = "lidar_camera_fusion.mp4"
iio.imwrite(output_video, frames, fps=10, codec="libx264")
print(f"\nVideo saved: {output_video}  ({num_frames} frames, 10 fps)")

## Interactive 3D Point Cloud Viewer

Static scatter plots can't capture the 3D structure of a LiDAR scan the way an interactive viewer like rviz can. As a notebook-friendly alternative, we use **plotly** to render the camera-colorized point cloud with full orbit, zoom, and pan controls. Each LiDAR point that projects onto the camera image gets the RGB color of its corresponding pixel.

Drag to orbit, scroll to zoom, shift-drag to pan.

In [ ]:
%pip install plotly --quiet

import plotly.graph_objects as go

# --- Colorize: project each LiDAR point and grab its camera pixel color ---
pts_3d_all = velo[:, :3]
front = pts_3d_all[:, 0] > 0
pts_front = pts_3d_all[front]

pts_hom_all = np.hstack([pts_front, np.ones((len(pts_front), 1))])
proj_all = P2 @ R0_ext @ T_cam_velo @ pts_hom_all.T
z_all = proj_all[2, :]
u_all = proj_all[0, :] / z_all
v_all = proj_all[1, :] / z_all

h, w = img.shape[:2]
in_img = (z_all > 0) & (u_all >= 0) & (u_all < w) & (v_all >= 0) & (v_all < h)

pts_visible = pts_front[in_img]
colors_rgb = img[v_all[in_img].astype(int), u_all[in_img].astype(int)]

# Subsample for plotly performance (~15k points renders smoothly)
step = max(1, len(pts_visible) // 15000)
pts_sub = pts_visible[::step]
colors_sub = colors_rgb[::step]

color_strs = [f"rgb({r},{g},{b})" for r, g, b in colors_sub]

fig = go.Figure(data=[go.Scatter3d(
    x=pts_sub[:, 0],
    y=pts_sub[:, 1],
    z=pts_sub[:, 2],
    mode="markers",
    marker=dict(size=1.5, color=color_strs, opacity=0.8),
    hovertemplate="X: %{x:.1f}m<br>Y: %{y:.1f}m<br>Z: %{z:.1f}m<extra></extra>"
)])

fig.update_layout(
    title="Interactive 3D LiDAR Point Cloud (colored by camera image)",
    scene=dict(
        xaxis_title="X — forward (m)",
        yaxis_title="Y — left (m)",
        zaxis_title="Z — up (m)",
        aspectmode="data",
        camera=dict(
            eye=dict(x=-0.8, y=-1.2, z=0.6),
            up=dict(x=0, y=0, z=1)
        ),
    ),
    width=950,
    height=650,
    margin=dict(l=0, r=0, t=40, b=0),
)

fig.show()

print(f"Displaying {len(pts_sub):,} points (subsampled from {len(pts_visible):,} camera-visible points)")

### Semantic Segmentation Point Cloud

Same 3D point cloud, but now each point is colored by its **semantic class** from SegFormer. This is the core idea behind "point painting" — transferring 2D image understanding onto 3D geometry so that downstream modules (tracking, planning) can reason about *what* objects are in 3D space, not just *where* they are.

In [ ]:
from scipy.ndimage import uniform_filter

# Smooth the segmentation mask with a local majority vote (5x5 window).
# For each class, compute the fraction of the window that belongs to it,
# then pick the class with the highest fraction at each pixel.
num_classes = seg_mask.max() + 1
class_votes = np.zeros((num_classes, *seg_mask.shape), dtype=np.float32)
for c in range(num_classes):
    class_votes[c] = uniform_filter((seg_mask == c).astype(np.float32), size=5)
seg_mask_smooth = class_votes.argmax(axis=0)

# Cityscapes color palette (RGB) — one color per class
CITYSCAPES_COLORS = [
    (128,  64, 128),  # road
    (244,  35, 232),  # sidewalk
    ( 70,  70,  70),  # building
    (102, 102, 156),  # wall
    (190, 153, 153),  # fence
    (153, 153, 153),  # pole
    (250, 170,  30),  # traffic light
    (220, 220,   0),  # traffic sign
    (107, 142,  35),  # vegetation
    (152, 251, 152),  # terrain
    ( 70, 130, 180),  # sky
    (220,  20,  60),  # person
    (255,   0,   0),  # rider
    (  0,   0, 142),  # car
    (  0,   0,  70),  # truck
    (  0,  60, 100),  # bus
    (  0,  80, 100),  # train
    (  0,   0, 230),  # motorcycle
    (119,  11,  32),  # bicycle
]

# Look up the semantic class for each visible LiDAR point (using smoothed mask)
u_px = u_all[in_img].astype(int)
v_px = v_all[in_img].astype(int)
point_classes = seg_mask_smooth[v_px, u_px]

# Map class IDs to colors (subsampled to match pts_sub)
seg_colors = np.array([CITYSCAPES_COLORS[c] for c in point_classes[::step]])
seg_color_strs = [f"rgb({r},{g},{b})" for r, g, b in seg_colors]

# Build hover labels with class names
point_labels = [CITYSCAPES_LABELS[c] for c in point_classes[::step]]

fig_seg = go.Figure(data=[go.Scatter3d(
    x=pts_sub[:, 0],
    y=pts_sub[:, 1],
    z=pts_sub[:, 2],
    mode="markers",
    marker=dict(size=1.5, color=seg_color_strs, opacity=0.8),
    text=point_labels,
    hovertemplate="X: %{x:.1f}m<br>Y: %{y:.1f}m<br>Z: %{z:.1f}m<br>Class: %{text}<extra></extra>"
)])

fig_seg.update_layout(
    title="Interactive 3D LiDAR Point Cloud (colored by semantic class)",
    scene=dict(
        xaxis_title="X — forward (m)",
        yaxis_title="Y — left (m)",
        zaxis_title="Z — up (m)",
        aspectmode="data",
        camera=dict(
            eye=dict(x=-0.8, y=-1.2, z=0.6),
            up=dict(x=0, y=0, z=1)
        ),
    ),
    width=950,
    height=650,
    margin=dict(l=0, r=0, t=40, b=0),
)

fig_seg.show()

# Print legend
print("Class legend:")
visible_classes = sorted(set(point_classes[::step]))
for c in visible_classes:
    count = (point_classes[::step] == c).sum()
    print(f"  {CITYSCAPES_LABELS[c]:15s}  ({count:,} points)")

## Summary

In this notebook we built the geometric foundation for camera–LiDAR fusion:

1. **Pinhole camera model** — decomposed KITTI's calibration into intrinsic matrix **K** and extrinsic transforms, with explicit coordinate-frame conventions (Velodyne vs Camera vs Image)
2. **3D → 2D forward projection** — applied `P2 · R0_rect · Tr_velo_to_cam · X` to project ~100k LiDAR points onto the camera image, colored by depth
3. **Synthetic overlay** — created a ground-plane grid and virtual vehicle bounding box in 3D and verified the projection places them physically where expected
4. **Semantic segmentation + back-projection** — used SegFormer to identify the drivable surface, then back-projected road boundary pixels to the ground plane to recover the road outline in metric bird's-eye-view coordinates
5. **Calibration sensitivity** — demonstrated that a 1° pitch error causes pixel drift that grows with depth, motivating the need for precise calibration
6. **Interactive 3D viewer** — rendered the camera-colorized LiDAR point cloud in an interactive plotly viewer, connecting the 2D image colors to the 3D spatial structure

### Key Takeaways

- **Forward projection loses depth** — a single camera image cannot recover 3D without an additional constraint or sensor
- **Back-projection requires a constraint** — the ground-plane assumption lets us invert the projection, but only for points on the plane; identifying *which* pixels are ground required a segmentation model
- **Calibration errors compound with distance** — small angular errors produce large pixel offsets at range, making extrinsic calibration critical for any fusion pipeline
- **LiDAR provides the depth that cameras lack** — fusing the two is more powerful than either alone (explored further in N3)

### What's Next

- **N2** — The Kalman filter: recursive state estimation from noisy measurements
- **N3** — EKF sensor fusion: combining complementary sensors through a nonlinear motion model
- **N4** — Multi-object tracking: upgrading per-frame YOLO detections into persistent tracked objects

### References
- [KITTI Vision Benchmark Suite](http://www.cvlibs.net/datasets/kitti/)
- [pykitti documentation](https://github.com/utiasSTARS/pykitti)
- [SegFormer (Xie et al., 2021)](https://arxiv.org/abs/2105.15203) — the semantic segmentation model used for road detection
- Hartley & Zisserman — *Multiple View Geometry in Computer Vision* (the standard reference for projection geometry)